# 📚 04_高级应用与生产

> **核心能力**：SQL 高级语法 (CTE/窗口函数)、数据库操作 (游标/CTAS)、自动化处理 (os 库)
>
> **开局心法**：从 Pandas 升级到 SQL 层面的思维；用游标向数据库下指令；用 os 库批量处理文件。这是数据分析从"个人玩具"到"生产级别"的关键跨越。

---

## 1️⃣ SQL 心法：CTE 神级连招 (`WITH`)

> **大白话口诀**：告别俄罗斯套娃（嵌套子查询）。只要你想按步骤做数据清洗，就在 SQL 的开头用 `WITH` 建立备菜盘。
> 它本质上就等于你在 **Pandas 里新建好几个临时 DataFrame 变量 (`df1 = ...`, `df2 = ...`)** 然后拼在一起！

### 📖 【SQL 视角：流水线写法】

```sql
WITH 
-- 🔪 备菜盘 A（等同于 Pandas 里的 df1 = ...）
Total_Sales AS (
    SELECT 用户ID, SUM(金额) AS 消费总额
    FROM orders
    GROUP BY 用户ID
),

-- 🔪 备菜盘 B（基于上面的 A 继续操作，等同于 df2 = ...）
Top_VIPs AS (
    SELECT 用户ID 
    FROM Total_Sales
    ORDER BY 消费总额 DESC
    LIMIT 2
)

-- 🍳 热油下锅（等同于 Pandas 最后的 pd.merge 动作）
SELECT o.* 
FROM orders o 
INNER JOIN Top_VIPs v ON o.用户ID = v.用户ID;
```

### 【Pandas 镜像对应】

```python
# 对应备菜盘 A
total_sales = df.groupby('用户ID')['金额'].sum().reset_index()
total_sales.columns = ['用户ID', '消费总额']

# 对应备菜盘 B
top_vips = total_sales.sort_values(by='消费总额', ascending=False).head(2)

# 对应最终下锅匹配 (注意右表加了双框！保尊严！)
final_result = pd.merge(df, top_vips[['用户ID']], on='用户ID', how='inner')
```

**关键点**：CTE 让复杂逻辑分步骤执行，易于调试和维护。

---
## 2️⃣ SQL 窗口函数：上帝视角的统计

> **大白话口诀**：比 GROUP BY 更强大。GROUP BY 会"折叠"表（一组一行），但窗口函数不折叠，直接在每一行旁边加一列"组内统计"。

### 【常用窗口函数模板】

```sql
SELECT 
    *,
    -- 1. 📊 [求全市场占比]
    ROUND(产值 * 100.0 / SUM(产值) OVER(), 2) AS 市场占比,
    
    -- 2. 👥 [求自己在各自品牌的内部排名]
    DENSE_RANK() OVER(PARTITION BY 品牌 ORDER BY 销量 DESC) AS 组内销冠排名,
    
    -- 3. ❄️ [滚雪球/累计求和]
    SUM(营业额) OVER(ORDER BY 营业额 DESC) AS 累计占领金额,
    
    -- 4. 📈 [移动平均（当前行及前 2 行）]
    AVG(销售额) OVER(PARTITION BY 品牌 ORDER BY 日期 ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS 三日均价
FROM orders;
```

### 【SQL vs Pandas 对照表】

| SQL 写法 | Pandas 写法 | 说明 |
|---------|-----------|------|
| `SUM(...) OVER()` | `.sum()` 或 `transform('sum')` | 全表求和 |
| `SUM(...) OVER(PARTITION BY 品牌)` | `.groupby('品牌').transform('sum')` | 分组求和并广播 |
| `RANK() OVER(ORDER BY ...)` | `.rank()` | 排名 |
| `DENSE_RANK() OVER(...)` | `.rank(method='dense')` | 密集排名（1,1,3） |
| `ROW_NUMBER() OVER(...)` | `.cumcount()` | 行号 |

---
## 3️⃣ 数据库操作：游标 (Cursor) 与 CTAS 建表

> **大白话口诀**：
> *   `Cursor` = 你派进数据库干活的"监工"，负责传递 SQL 指令。
> *   `CTAS` (Create Table As Select) = 把查出来的结果直接"固化"成一张新表，存在数据库里，而不是只留在 Python 内存里。

In [ ]:
import psycopg2
import pandas as pd

# 1. 🔑 [连大门]：建立数据库连接
conn = psycopg2.connect(
    host='localhost',
    port='5432',
    database='your_database',
    user='your_user',
    password='your_password'
)

# 2. 👷 [派监工]：创建游标对象
cur = conn.cursor()

# 3. 🔨 [简单查询]：执行查询并读取结果
cur.execute("SELECT * FROM orders LIMIT 5")
results = cur.fetchall()  # 拿出来
print(results)

# 4. 🚚 [更简单的办法]：直接用 Pandas 读数据库
df = pd.read_sql("SELECT * FROM orders", conn)

### CTAS 建表：把查询结果固化成新表

```python
# 【场景】你在 Python 里处理数据后，想直接存回数据库成新表，不走中间 CSV

query_ctas = """
DROP TABLE IF EXISTS analysis.report_temp;

CREATE TABLE analysis.report_temp AS
SELECT 
    地区,
    ROUND(AVG(销售额), 2) AS 平均销售额,
    COUNT(*) AS 订单数,
    MAX(日期) AS 最后更新
FROM raw_data.orders
WHERE 日期 >= '2026-01-01'
GROUP BY 地区;
"""

cur.execute(query_ctas)
conn.commit()  # ⚠️ 重要！必须提交，否则数据库不认账

# 后续可以从 report_temp 直接读取
df_report = pd.read_sql("SELECT * FROM analysis.report_temp", conn)
```

In [ ]:
# 5. 🧹 [打扫战场]：用完记得关门关灯
cur.close()
conn.close()

### 🚨 坑点：数据库游标的常见错误

#### 坑点 1：忘记 commit()，数据没有真正保存

```python
# ❌ 常见错误
cur.execute("CREATE TABLE xxx AS SELECT ...")
# 然后关闭连接，表没了！

# ✅ 正确做法
cur.execute("CREATE TABLE xxx AS SELECT ...")
conn.commit()  # ← 不能忘！
```

---

#### 坑点 2：SELECT 后忘记 fetch，结果没拿出来

```python
# ❌ 只执行了，但没拿结果
cur.execute("SELECT * FROM orders")
# 然后代码继续运行，这条查询的结果丢失了

# ✅ 正确做法
cur.execute("SELECT * FROM orders")
results = cur.fetchall()  # 拿出所有行
# 或
results = cur.fetchone()  # 只拿第一行
# 或（推荐）
df = pd.read_sql("SELECT * FROM orders", conn)  # 直接变成 DataFrame
```

---
## 4️⃣ 自动化办公：批量处理文件夹下的文件 (`os` 库)

> **大白话口诀**：先用 `os` 挨个点名列出文件名，再根据"后缀名"过滤，最后配合 Pandas 的 `read` 和 `to` 实现"格式大挪移"。

In [ ]:
import os
import pandas as pd

# 💡 技巧：自动获取当前脚本/Notebook 所在的文件夹
# 在脚本中用: os.path.dirname(os.path.abspath(__file__))
target_dir = r"C:\你的数据文件夹路径"

# ================ 场景 1：批量 Excel → CSV ================
def batch_convert_xlsx_to_csv(path):
    # 拿到所有 xlsx 文件名，并排除正在打开的临时文件 (~$)
    files = [f for f in os.listdir(path) if f.endswith(".xlsx") and not f.startswith("~$")]
    
    for filename in files:
        file_path = os.path.join(path, filename)
        
        # 📂 1. 读取 Excel
        df = pd.read_excel(file_path)
        
        # 🔗 2. 构造 CSV 路径 (把后缀换成 .csv)
        pure_name = os.path.splitext(filename)[0]
        csv_path = os.path.join(path, f"{pure_name}.csv")
        
        # ✍️ 3. 保存 CSV (编码选 utf-8-sig 才能在 Excel 里看中文)
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"✅ 已完成: {filename} -> {pure_name}.csv")

# batch_convert_xlsx_to_csv(target_dir) # 取消注释运行

In [ ]:
# ================ 场景 2：批量合并多个 CSV ================
def batch_merge_csv(path):
    # 拿到所有 csv 文件
    files = [f for f in os.listdir(path) if f.endswith(".csv")]
    
    dfs = []
    for filename in files:
        file_path = os.path.join(path, filename)
        df = pd.read_csv(file_path, encoding='utf-8-sig')
        dfs.append(df)
        print(f"📖 已读取: {filename}")
    
    # 纵向拼接（行堆叠）
    merged_df = pd.concat(dfs, axis=0, ignore_index=True)
    
    # 保存合并后的文件
    output_path = os.path.join(path, "_MERGED.csv")
    merged_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"✅ 合并完成！共 {len(merged_df)} 行。保存到 {output_path}")

# batch_merge_csv(target_dir)

### 【常用 os 操作速查】

| 操作 | 代码 | 说明 |
|------|------|------|
| 列出文件夹所有文件 | `os.listdir(path)` | 返回列表 |
| 拼接路径 | `os.path.join(path, filename)` | 跨平台兼容 |
| 提取文件名（不带后缀） | `os.path.splitext(filename)[0]` | 返回 (名字, 后缀) 元组 |
| 提取后缀 | `os.path.splitext(filename)[1]` | 返回 ".csv" |
| 检查文件是否存在 | `os.path.exists(path)` | True/False |
| 检查是否为文件 | `os.path.isfile(path)` | True/False |
| 检查是否为文件夹 | `os.path.isdir(path)` | True/False |
| 删除文件 | `os.remove(path)` | ⚠️ 不可恢复 |
| 创建文件夹 | `os.makedirs(path)` | 支持嵌套创建 |
| 获取文件大小 | `os.path.getsize(path)` | 字节数 |

### 🚨 坑点：os 操作的陷阱

#### 坑点 1：os.listdir() 包括了文件夹，需要过滤

```python
# ❌ 如果路径里有子文件夹
files = os.listdir(path)  # 结果里既有文件，也有文件夹名

# ✅ 只要文件
files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
```

---

#### 坑点 2：Windows 路径反斜杠导致转义问题

```python
# ❌ 容易出错
path = "C:\Users\data"  # 变成了特殊字符转义！

# ✅ 用原始字符串
path = r"C:\Users\data"  # ✓
# 或
path = "C:/Users/data"  # ✓（正斜杠 Python 也认）
```

---
## 【综合案例】从数据库读、处理、再写回的完整流程

```python
import psycopg2
import pandas as pd
import os

# Step 1: 从数据库读数据
conn = psycopg2.connect(...)
df = pd.read_sql("""
    SELECT * FROM raw_orders 
    WHERE date >= '2026-01-01'
""", conn)

# Step 2: 在 Pandas 里处理
df['交易月'] = pd.to_datetime(df['日期']).dt.to_period('M')
summary = df.groupby(['交易月', '城市']).agg({
    '销售额': 'sum',
    '订单数': 'count'
}).reset_index()

# Step 3: 写回数据库（CTAS）
cur = conn.cursor()
cur.execute("""
    DROP TABLE IF EXISTS analysis.summary;
    CREATE TABLE analysis.summary AS
    SELECT * FROM ...;  # 从上面的 summary 导出 SQL
""")
conn.commit()
conn.close()

# Step 4: 或者导出到本地 CSV
summary.to_csv(r'C:\output\summary.csv', index=False, encoding='utf-8-sig')
```